In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [13]:
customer = pd.read_excel(
    r"E:\CODING_ALL\Data Analytics\projrcts\RetailIQ\data\processed\customer_master.xlsx",
    sheet_name="processed"
)

df = customer.copy()

In [14]:
print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
df.info()

display(df.head())

Dataset shape: (25000, 12)

Column names:
['customer_id', 'customer_name', 'customer_age', 'age_range', 'gender', 'customer_segment', 'customer_city', 'customer_state', 'customer_country', 'region', 'customer_postal_code', 'customer_acquisition_cost']

Data types:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   customer_id                25000 non-null  object 
 1   customer_name              25000 non-null  object 
 2   customer_age               25000 non-null  int64  
 3   age_range                  25000 non-null  object 
 4   gender                     25000 non-null  object 
 5   customer_segment           25000 non-null  object 
 6   customer_city              25000 non-null  object 
 7   customer_state             25000 non-null  object 
 8   customer_country           25000 non-null  object 
 9   regio

,customer_id,customer_name,customer_age,age_range,gender,customer_segment,customer_city,customer_state,customer_country,region,customer_postal_code,customer_acquisition_cost
0,CUST-000001,Donna Miller,54,50-60,Female,Consumer,Laurenland,Dubai,UAE,South,11253,13.58
1,CUST-000002,Joseph James,61,60-70,Male,Consumer,Lake Peter,Lower Saxony,Germany,North,47778,27.45
2,CUST-000003,Daniel Smith,47,40-50,Male,Consumer,Seanview,North Rhine,Germany,West,13900,31.53
3,CUST-000004,David Lopez,30,30-40,Male,VIP,Hawkinshaven,Pennsylvania,USA,East,53984,16.02
4,CUST-000005,Dakota Davis,55,50-60,Female,Consumer,New Carlymouth,Florida,USA,South,50994,41.34


In [15]:
df.columns = df.columns.str.strip()

missing = pd.DataFrame({
    "Missing_Count": df.isnull().sum(),
    "Missing_Percentage": (df.isnull().mean() * 100).round(2)
})

display(missing.sort_values("Missing_Percentage", ascending=False))

,Missing_Count,Missing_Percentage
customer_id,0,0.0
customer_name,0,0.0
customer_age,0,0.0
age_range,0,0.0
gender,0,0.0
customer_segment,0,0.0
customer_city,0,0.0
customer_state,0,0.0
customer_country,0,0.0
region,0,0.0


In [16]:
print("Exact duplicate rows:", df.duplicated().sum())
print("Missing customer IDs:", df["customer_id"].isna().sum())
print("Unique customer IDs:", df["customer_id"].nunique())
print("Duplicate customer IDs:", df["customer_id"].duplicated().sum())

display(df[df["customer_id"].duplicated(keep=False)].head(10))

Exact duplicate rows: 0
Missing customer IDs: 0
Unique customer IDs: 25000
Duplicate customer IDs: 0


,customer_id,customer_name,customer_age,age_range,gender,customer_segment,customer_city,customer_state,customer_country,region,customer_postal_code,customer_acquisition_cost


In [17]:
numeric_cols = [
    "customer_age",
    "customer_acquisition_cost"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

display(df[numeric_cols].describe().T)

,count,mean,std,min,25%,50%,75%,max
customer_age,25000.0,45.90784,16.454286,18.00,32.0000,46.00,60.00,74.00
customer_acquisition_cost,25000.0,42.15975,21.692648,5.01,23.2975,41.98,61.13,79.99


In [18]:
checks = {
    "Age below 0": df["customer_age"] < 0,
    "Age above 100": df["customer_age"] > 100,
    "Negative acquisition cost": df["customer_acquisition_cost"] < 0
}

for issue, condition in checks.items():
    print(f"{issue}: {condition.sum()}")

Age below 0: 0
Age above 100: 0
Negative acquisition cost: 0


In [19]:
print("Age ranges:")
print(df["age_range"].value_counts(dropna=False))

display(
    df[["customer_age", "age_range"]].drop_duplicates().sort_values(
        "customer_age", na_position="last"
    ).head(30)
)

Age ranges:
age_range
40-50    4415
20-30    4401
30-40    4387
50-60    4324
60-70    4322
70-80    2230
10-20     921
Name: count, dtype: int64


,customer_age,age_range
117,18,10-20
78,19,10-20
84,20,20-30
76,21,20-30
47,22,20-30
130,23,20-30
102,24,20-30
24,25,20-30
46,26,20-30
6,27,20-30


In [20]:
categorical_cols = [
    "gender",
    "customer_segment",
    "customer_city",
    "customer_state",
    "customer_country",
    "region"
]

for col in categorical_cols:
    print(f"\n--- {col} ---")
    print("Unique values:", df[col].nunique())
    print(df[col].value_counts(dropna=False).head(10))


--- gender ---
Unique values: 3
gender
Female        12048
Male          11976
Non-Binary      976
Name: count, dtype: int64

--- customer_segment ---
Unique values: 4
customer_segment
Consumer    13638
Premium      6301
VIP          2542
Business     2519
Name: count, dtype: int64

--- customer_city ---
Unique values: 15555
customer_city
Lake Michael     30
South Michael    30
East Michael     23
Port Michael     21
South James      20
New David        19
New Robert       18
West Michael     17
North David      17
New John         17
Name: count, dtype: int64

--- customer_state ---
Unique values: 38
customer_state
Pennsylvania      1632
California        1530
Georgia           1530
New York          1517
Ohio              1496
Michigan          1483
Texas             1463
Illinois          1439
North Carolina    1434
Florida           1401
Name: count, dtype: int64

--- customer_country ---
Unique values: 7
customer_country
USA          14925
UK            3701
Germany       2046
Ca

In [21]:
df["customer_postal_code"] = df["customer_postal_code"].astype("string").str.strip()

print("Missing postal codes:", df["customer_postal_code"].isna().sum())
print("Unique postal codes:", df["customer_postal_code"].nunique())

Missing postal codes: 0
Unique postal codes: 22183


In [24]:
df.to_csv("cleaned_customers.csv", index=False)
print("Saved!")

Saved!
